<a href="https://colab.research.google.com/github/ayushhh026/RuppeRisk/blob/main/notebooks/Model07_Installments_Feature_Engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Mount Drive and import required libraries
from google.colab import drive
drive.mount('/content/drive')

import os
import gc
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

from sklearn.metrics import roc_auc_score, average_precision_score
from xgboost import XGBClassifier

Mounted at /content/drive


In [2]:
# Define data and results paths
DATA_PATH = "/content/drive/MyDrive/datasets/raw/"
RESULTS_PATH = "/content/drive/MyDrive/RupeeRisk/"
os.makedirs(RESULTS_PATH, exist_ok=True)

In [3]:
# Load application_train.csv
application = pd.read_csv(DATA_PATH + "application_train.csv")
print("Application shape:", application.shape)

Application shape: (307511, 122)


In [4]:
# Load previous_application.csv
prev = pd.read_csv(DATA_PATH + "previous_application.csv")
print("Previous application shape:", prev.shape)
# Load bureau.csv
bureau = pd.read_csv(DATA_PATH + "bureau.csv")
print("Bureau shape:", bureau.shape)
# Load bureau_balance.csv
bureau_balance = pd.read_csv(DATA_PATH + "bureau_balance.csv")
print("Bureau balance shape:", bureau_balance.shape)
# Load POS_CASH_balance.csv with compact dtypes
pos_dtype = {
    "SK_ID_PREV": np.uint32,
    "SK_ID_CURR": np.uint32,
    "MONTHS_BALANCE": np.int32,
    "SK_DPD": np.int32,
    "SK_DPD_DEF": np.int32,
    "CNT_INSTALMENT": np.float32,
    "CNT_INSTALMENT_FUTURE": np.float32
}
pos = pd.read_csv(DATA_PATH + "POS_CASH_balance.csv", dtype=pos_dtype)
print("POS_CASH shape:", pos.shape)
# Load installments_payments.csv with compact dtypes
install_dtype = {
    "SK_ID_PREV": np.uint32,
    "SK_ID_CURR": np.uint32,
    "NUM_INSTALMENT_NUMBER": np.int32,
    "NUM_INSTALMENT_VERSION": np.float32,
    "DAYS_INSTALMENT": np.float32,
    "DAYS_ENTRY_PAYMENT": np.float32,
    "AMT_INSTALMENT": np.float32,
    "AMT_PAYMENT": np.float32
}
installments = pd.read_csv(DATA_PATH + "installments_payments.csv", dtype=install_dtype)
print("Installments shape:", installments.shape)

Previous application shape: (1670214, 37)
Bureau shape: (1716428, 17)
Bureau balance shape: (27299925, 3)
POS_CASH shape: (10001358, 8)
Installments shape: (13605401, 8)


In [5]:
# Reapply DAYS_EMPLOYED sentinel fix, EXT_SOURCE flags, ratios, EXT_SOURCE combos
application["DAYS_EMPLOYED_ANOM"] = (application["DAYS_EMPLOYED"] == 365243).astype(int)
application["DAYS_EMPLOYED"] = application["DAYS_EMPLOYED"].replace(365243, np.nan)

application["EXT_SOURCE_1_MISSING"] = application["EXT_SOURCE_1"].isna().astype(int)
application["EXT_SOURCE_3_MISSING"] = application["EXT_SOURCE_3"].isna().astype(int)

application["AGE_YEARS"] = -application["DAYS_BIRTH"] / 365.25
application["EMPLOYMENT_YEARS"] = -application["DAYS_EMPLOYED"] / 365.25

application["CREDIT_INCOME_RATIO"] = application["AMT_CREDIT"] / application["AMT_INCOME_TOTAL"]
application["ANNUITY_INCOME_RATIO"] = application["AMT_ANNUITY"] / application["AMT_INCOME_TOTAL"]
application["ANNUITY_CREDIT_RATIO"] = application["AMT_ANNUITY"] / application["AMT_CREDIT"]
application["GOODS_CREDIT_RATIO"] = application["AMT_GOODS_PRICE"] / application["AMT_CREDIT"]
application["EMPLOYMENT_AGE_RATIO"] = application["EMPLOYMENT_YEARS"] / application["AGE_YEARS"]

ext_cols = ["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]
application["EXT_SOURCE_MEAN"] = application[ext_cols].mean(axis=1)
application["EXT_SOURCE_MIN"] = application[ext_cols].min(axis=1)
application["EXT_SOURCE_MAX"] = application[ext_cols].max(axis=1)
application["EXT_SOURCE_STD"] = application[ext_cols].std(axis=1)

application["EXT_SOURCE_1_2"] = application["EXT_SOURCE_1"] * application["EXT_SOURCE_2"]
application["EXT_SOURCE_1_3"] = application["EXT_SOURCE_1"] * application["EXT_SOURCE_3"]
application["EXT_SOURCE_2_3"] = application["EXT_SOURCE_2"] * application["EXT_SOURCE_3"]

print("MODEL02 features recreated.")

MODEL02 features recreated.


In [6]:
# Rebuild previous_application aggregations: counts, financials, ratios, status rates, timing, payments
prev_count = (
    prev.groupby("SK_ID_CURR")
    .size()
    .rename("PREV_APPLICATION_COUNT")
    .reset_index()
)

financial_agg = (
    prev.groupby("SK_ID_CURR")
    .agg({
        "AMT_CREDIT": ["mean", "max", "sum"],
        "AMT_APPLICATION": ["mean", "max", "sum"],
        "AMT_ANNUITY": ["mean", "max", "sum"],
        "AMT_GOODS_PRICE": ["mean", "max", "sum"],
        "AMT_DOWN_PAYMENT": ["mean", "max"],
        "RATE_DOWN_PAYMENT": ["mean", "max"]
    })
)
financial_agg.columns = ["PREV_" + col[0] + "_" + col[1].upper() for col in financial_agg.columns]
financial_agg = financial_agg.reset_index()

prev["PREV_CREDIT_APPL_RATIO"] = prev["AMT_CREDIT"] / prev["AMT_APPLICATION"].replace(0, np.nan)
prev["PREV_CREDIT_APPL_DIFF"] = prev["AMT_CREDIT"] - prev["AMT_APPLICATION"]

relationship_agg = (
    prev.groupby("SK_ID_CURR")
    .agg({
        "PREV_CREDIT_APPL_RATIO": ["mean", "max"],
        "PREV_CREDIT_APPL_DIFF": ["mean", "max"]
    })
)
relationship_agg.columns = ["PREV_" + col[0] + "_" + col[1].upper() for col in relationship_agg.columns]
relationship_agg = relationship_agg.reset_index()

prev["PREV_APPROVED"] = (prev["NAME_CONTRACT_STATUS"] == "Approved").astype(int)
prev["PREV_REFUSED"] = (prev["NAME_CONTRACT_STATUS"] == "Refused").astype(int)
prev["PREV_CANCELED"] = (prev["NAME_CONTRACT_STATUS"] == "Canceled").astype(int)
prev["PREV_UNUSED"] = (prev["NAME_CONTRACT_STATUS"] == "Unused offer").astype(int)

status_agg = (
    prev.groupby("SK_ID_CURR")
    .agg({
        "PREV_APPROVED": "sum",
        "PREV_REFUSED": "sum",
        "PREV_CANCELED": "sum",
        "PREV_UNUSED": "sum"
    })
    .reset_index()
)
status_agg = status_agg.rename(columns={
    "PREV_APPROVED": "PREV_APPROVED_COUNT",
    "PREV_REFUSED": "PREV_REFUSED_COUNT",
    "PREV_CANCELED": "PREV_CANCELED_COUNT",
    "PREV_UNUSED": "PREV_UNUSED_COUNT"
})

status_agg = status_agg.merge(
    prev_count[["SK_ID_CURR", "PREV_APPLICATION_COUNT"]],
    on="SK_ID_CURR", how="left"
)
status_agg["PREV_APPROVAL_RATE"] = status_agg["PREV_APPROVED_COUNT"] / status_agg["PREV_APPLICATION_COUNT"]
status_agg["PREV_REFUSAL_RATE"] = status_agg["PREV_REFUSED_COUNT"] / status_agg["PREV_APPLICATION_COUNT"]
status_agg["PREV_CANCELLATION_RATE"] = status_agg["PREV_CANCELED_COUNT"] / status_agg["PREV_APPLICATION_COUNT"]
status_agg = status_agg.drop(columns=["PREV_APPLICATION_COUNT"])

decision_agg = (
    prev.groupby("SK_ID_CURR")
    .agg({"DAYS_DECISION": ["min", "max", "mean"]})
)
decision_agg.columns = ["PREV_DAYS_DECISION_" + col[1].upper() for col in decision_agg.columns]
decision_agg = decision_agg.reset_index()

payment_agg = (
    prev.groupby("SK_ID_CURR")
    .agg({"CNT_PAYMENT": ["mean", "max", "sum"]})
)
payment_agg.columns = ["PREV_CNT_PAYMENT_" + col[1].upper() for col in payment_agg.columns]
payment_agg = payment_agg.reset_index()

prev_features = prev_count.copy()
prev_features = prev_features.merge(financial_agg, on="SK_ID_CURR", how="left")
prev_features = prev_features.merge(relationship_agg, on="SK_ID_CURR", how="left")
prev_features = prev_features.merge(status_agg, on="SK_ID_CURR", how="left")
prev_features = prev_features.merge(decision_agg, on="SK_ID_CURR", how="left")
prev_features = prev_features.merge(payment_agg, on="SK_ID_CURR", how="left")

print("Previous-application features:", prev_features.shape)

Previous-application features: (338857, 35)


In [7]:
# Rebuild bureau aggregations: counts, financials, overdue severity, status counts, credit types, timing
bureau_count = (
    bureau.groupby("SK_ID_CURR")
    .size()
    .rename("BUREAU_CREDIT_COUNT")
    .reset_index()
)

bureau_financial_agg = (
    bureau.groupby("SK_ID_CURR")
    .agg({
        "AMT_CREDIT_SUM": ["mean", "max", "sum"],
        "AMT_CREDIT_SUM_DEBT": ["mean", "max", "sum"],
        "AMT_CREDIT_SUM_LIMIT": ["mean", "max"],
        "AMT_ANNUITY": ["mean"]
    })
)
bureau_financial_agg.columns = ["BUREAU_" + col[0] + "_" + col[1].upper() for col in bureau_financial_agg.columns]
bureau_financial_agg = bureau_financial_agg.reset_index()

bureau["BUREAU_OVERDUE_FLAG"] = (bureau["CREDIT_DAY_OVERDUE"] > 0).astype(int)

bureau_overdue_agg = (
    bureau.groupby("SK_ID_CURR")
    .agg({
        "CREDIT_DAY_OVERDUE": ["max"],
        "BUREAU_OVERDUE_FLAG": ["sum"],
        "AMT_CREDIT_SUM_OVERDUE": ["max", "sum"]
    })
    .reset_index()
)
bureau_overdue_agg.columns = [
    "SK_ID_CURR", "BUREAU_OVERDUE_DAYS_MAX", "BUREAU_OVERDUE_COUNT",
    "BUREAU_OVERDUE_AMOUNT_MAX", "BUREAU_OVERDUE_AMOUNT_SUM"
]

bureau_overdue_agg = bureau_overdue_agg.merge(
    bureau_count[["SK_ID_CURR", "BUREAU_CREDIT_COUNT"]],
    on="SK_ID_CURR", how="left"
)
bureau_overdue_agg["BUREAU_OVERDUE_RATIO"] = (
    bureau_overdue_agg["BUREAU_OVERDUE_COUNT"] / bureau_overdue_agg["BUREAU_CREDIT_COUNT"]
)
bureau_overdue_agg = bureau_overdue_agg.drop(columns=["BUREAU_CREDIT_COUNT"])

bureau["BUREAU_ACTIVE_FLAG"] = (bureau["CREDIT_ACTIVE"] == "Active").astype(int)
bureau["BUREAU_CLOSED_FLAG"] = (bureau["CREDIT_ACTIVE"] == "Closed").astype(int)
bureau["BUREAU_SOLD_FLAG"] = (bureau["CREDIT_ACTIVE"] == "Sold").astype(int)
bureau["BUREAU_BAD_DEBT_FLAG"] = (bureau["CREDIT_ACTIVE"] == "Bad debt").astype(int)

bureau_status_agg = (
    bureau.groupby("SK_ID_CURR")
    .agg({
        "BUREAU_ACTIVE_FLAG": "sum",
        "BUREAU_CLOSED_FLAG": "sum",
        "BUREAU_SOLD_FLAG": "sum",
        "BUREAU_BAD_DEBT_FLAG": "sum"
    })
    .reset_index()
)
bureau_status_agg = bureau_status_agg.rename(columns={
    "BUREAU_ACTIVE_FLAG": "BUREAU_ACTIVE_COUNT",
    "BUREAU_CLOSED_FLAG": "BUREAU_CLOSED_COUNT",
    "BUREAU_SOLD_FLAG": "BUREAU_SOLD_COUNT",
    "BUREAU_BAD_DEBT_FLAG": "BUREAU_BAD_DEBT_COUNT"
})

bureau_status_agg = bureau_status_agg.merge(
    bureau_count[["SK_ID_CURR", "BUREAU_CREDIT_COUNT"]],
    on="SK_ID_CURR", how="left"
)
bureau_status_agg["BUREAU_ACTIVE_RATIO"] = (
    bureau_status_agg["BUREAU_ACTIVE_COUNT"] / bureau_status_agg["BUREAU_CREDIT_COUNT"]
)
bureau_status_agg = bureau_status_agg.drop(columns=["BUREAU_CREDIT_COUNT"])

bureau_type_agg = (
    bureau.groupby("SK_ID_CURR")
    .agg({"CREDIT_TYPE": "nunique"})
    .reset_index()
)
bureau_type_agg = bureau_type_agg.rename(columns={"CREDIT_TYPE": "BUREAU_CREDIT_TYPE_COUNT"})

bureau["BUREAU_CREDIT_CARD_FLAG"] = (bureau["CREDIT_TYPE"] == "Credit card").astype(int)
bureau["BUREAU_MORTGAGE_FLAG"] = (bureau["CREDIT_TYPE"] == "Mortgage").astype(int)
bureau["BUREAU_MICROLOAN_FLAG"] = (bureau["CREDIT_TYPE"] == "Microloan").astype(int)

bureau_type_specific = (
    bureau.groupby("SK_ID_CURR")
    .agg({
        "BUREAU_CREDIT_CARD_FLAG": "sum",
        "BUREAU_MORTGAGE_FLAG": "sum",
        "BUREAU_MICROLOAN_FLAG": "sum"
    })
    .reset_index()
)
bureau_type_specific = bureau_type_specific.rename(columns={
    "BUREAU_CREDIT_CARD_FLAG": "BUREAU_CREDIT_CARD_COUNT",
    "BUREAU_MORTGAGE_FLAG": "BUREAU_MORTGAGE_COUNT",
    "BUREAU_MICROLOAN_FLAG": "BUREAU_MICROLOAN_COUNT"
})

bureau_type_agg = bureau_type_agg.merge(bureau_type_specific, on="SK_ID_CURR", how="left")

bureau_timing_agg = (
    bureau.groupby("SK_ID_CURR")
    .agg({
        "DAYS_CREDIT": ["min", "max", "mean"],
        "DAYS_CREDIT_UPDATE": ["mean"]
    })
)
bureau_timing_agg.columns = ["BUREAU_" + col[0] + "_" + col[1].upper() for col in bureau_timing_agg.columns]
bureau_timing_agg = bureau_timing_agg.reset_index()

bureau_prolong_agg = (
    bureau.groupby("SK_ID_CURR")
    .agg({"CNT_CREDIT_PROLONG": "sum"})
    .reset_index()
)
bureau_prolong_agg = bureau_prolong_agg.rename(columns={"CNT_CREDIT_PROLONG": "BUREAU_CREDIT_PROLONG_TOTAL"})

bureau_features = bureau_count.copy()
for block in [bureau_financial_agg, bureau_overdue_agg, bureau_status_agg, bureau_type_agg, bureau_timing_agg, bureau_prolong_agg]:
    bureau_features = bureau_features.merge(block, on="SK_ID_CURR", how="left")

print("Bureau features:", bureau_features.shape)

Bureau features: (305811, 30)


In [8]:
# Rebuild two-stage bureau_balance aggregation: monthly -> credit -> applicant
bureau_balance["BB_STATUS_0"] = (bureau_balance["STATUS"] == "0").astype(int)
bureau_balance["BB_STATUS_1"] = (bureau_balance["STATUS"] == "1").astype(int)
bureau_balance["BB_STATUS_2_PLUS"] = bureau_balance["STATUS"].isin(["2", "3", "4", "5"]).astype(int)
bureau_balance["BB_STATUS_C"] = (bureau_balance["STATUS"] == "C").astype(int)
bureau_balance["BB_STATUS_X"] = (bureau_balance["STATUS"] == "X").astype(int)

bb_credit = (
    bureau_balance.groupby("SK_ID_BUREAU")
    .agg({
        "MONTHS_BALANCE": ["count"],
        "BB_STATUS_0": "sum",
        "BB_STATUS_1": "sum",
        "BB_STATUS_2_PLUS": "sum",
        "BB_STATUS_C": "sum",
        "BB_STATUS_X": "sum"
    })
)
bb_credit.columns = ["BB_" + col[0] + "_" + col[1].upper() for col in bb_credit.columns]
bb_credit = bb_credit.reset_index()

bb_credit = bb_credit.rename(columns={
    "BB_BB_STATUS_0_SUM": "BB_STATUS_0_COUNT",
    "BB_BB_STATUS_1_SUM": "BB_STATUS_1_COUNT",
    "BB_BB_STATUS_2_PLUS_SUM": "BB_STATUS_2_PLUS_COUNT",
    "BB_BB_STATUS_C_SUM": "BB_STATUS_C_COUNT",
    "BB_BB_STATUS_X_SUM": "BB_STATUS_X_COUNT"
})

bb_credit["BB_DELINQUENCY_COUNT"] = bb_credit["BB_STATUS_1_COUNT"] + bb_credit["BB_STATUS_2_PLUS_COUNT"]
bb_credit["BB_DELINQUENCY_RATE"] = bb_credit["BB_DELINQUENCY_COUNT"] / bb_credit["BB_MONTHS_BALANCE_COUNT"]
bb_credit["BB_STATUS_2_PLUS_RATE"] = bb_credit["BB_STATUS_2_PLUS_COUNT"] / bb_credit["BB_MONTHS_BALANCE_COUNT"]
bb_credit["BB_EVER_DELINQUENT"] = (bb_credit["BB_DELINQUENCY_COUNT"] > 0).astype(int)
bb_credit["BB_EVER_SEVERE_DELINQUENCY"] = (bb_credit["BB_STATUS_2_PLUS_COUNT"] > 0).astype(int)

bureau_with_balance = bureau[["SK_ID_CURR", "SK_ID_BUREAU"]].merge(bb_credit, on="SK_ID_BUREAU", how="left")

bb_app = (
    bureau_with_balance.groupby("SK_ID_CURR")
    .agg({
        "BB_MONTHS_BALANCE_COUNT": ["mean", "max", "sum"],
        "BB_DELINQUENCY_COUNT": ["mean", "max", "sum"],
        "BB_DELINQUENCY_RATE": ["mean", "max"],
        "BB_STATUS_2_PLUS_RATE": ["mean", "max"],
        "BB_STATUS_1_COUNT": "sum",
        "BB_STATUS_2_PLUS_COUNT": "sum",
        "BB_STATUS_C_COUNT": "sum",
        "BB_STATUS_X_COUNT": "sum",
        "BB_EVER_DELINQUENT": "sum",
        "BB_EVER_SEVERE_DELINQUENCY": "sum"
    })
)
bb_app.columns = ["BB_" + col[0] + "_" + col[1].upper() for col in bb_app.columns]
bb_app = bb_app.reset_index()
bb_app.columns = [col.replace("BB_BB_", "BB_") for col in bb_app.columns]

credits_with_balance = (
    bureau_with_balance.groupby("SK_ID_CURR")["BB_MONTHS_BALANCE_COUNT"]
    .count()
    .rename("BUREAU_CREDITS_WITH_BALANCE_HISTORY")
    .reset_index()
)
credits_with_balance = credits_with_balance.merge(
    bureau_count[["SK_ID_CURR", "BUREAU_CREDIT_COUNT"]],
    on="SK_ID_CURR", how="left"
)
credits_with_balance["BUREAU_BALANCE_COVERAGE_RATIO"] = (
    credits_with_balance["BUREAU_CREDITS_WITH_BALANCE_HISTORY"] / credits_with_balance["BUREAU_CREDIT_COUNT"]
)
credits_with_balance = credits_with_balance.drop(columns=["BUREAU_CREDIT_COUNT"])

bureau_balance_features = bb_app.merge(credits_with_balance, on="SK_ID_CURR", how="left")

print("Bureau balance features:", bureau_balance_features.shape)

Bureau balance features: (305811, 19)


In [9]:
# Merge application + previous_application + bureau + bureau_balance
application_model = application.copy()
application_model = application_model.merge(prev_features, on="SK_ID_CURR", how="left")
application_model = application_model.merge(bureau_features, on="SK_ID_CURR", how="left")
application_model = application_model.merge(bureau_balance_features, on="SK_ID_CURR", how="left")

print("Model05 base shape:", application_model.shape)

Model05 base shape: (307511, 220)


In [10]:
# Rebuild simplified POS_CASH features: overall aggregates, count, late rate, status share, completed rate
pos["LATE_POS"] = (pos["SK_DPD"] > 0).astype("int8")
pos["SK_DPD_RATIO"] = pos["SK_DPD"] / (pos["SK_DPD_DEF"] + 1)
pos["POS_REMAINING_INST_RATIO"] = pos["CNT_INSTALMENT_FUTURE"] / pos["CNT_INSTALMENT"].replace(0, np.nan)
pos["POS_INSTALLMENT_PROGRESS"] = 1 - pos["POS_REMAINING_INST_RATIO"]

pos_numeric_cols = [
    "MONTHS_BALANCE", "SK_DPD", "SK_DPD_DEF", "CNT_INSTALMENT", "CNT_INSTALMENT_FUTURE",
    "SK_DPD_RATIO", "POS_REMAINING_INST_RATIO", "POS_INSTALLMENT_PROGRESS"
]

pos_overall = (
    pos.groupby("SK_ID_CURR")[pos_numeric_cols]
    .agg(["min", "max", "mean", "sum", "var"])
)
pos_overall.columns = ["POS_" + col[0] + "_" + col[1].upper() for col in pos_overall.columns]
pos_overall = pos_overall.reset_index()

pos_count = (
    pos.groupby("SK_ID_CURR")
    .size()
    .rename("POS_COUNT")
    .reset_index()
)

pos_late_rate = (
    pos.groupby("SK_ID_CURR")["LATE_POS"]
    .mean()
    .rename("POS_LATE_RATE")
    .reset_index()
)

status_dummies = pd.get_dummies(
    pos[["SK_ID_CURR", "NAME_CONTRACT_STATUS"]],
    columns=["NAME_CONTRACT_STATUS"],
    dummy_na=True,
    dtype=np.float32
)
status_share = status_dummies.groupby("SK_ID_CURR").mean().reset_index()

status_share_cols = [col for col in status_share.columns if col != "SK_ID_CURR"]
status_share = status_share.rename(columns={col: "POS_" + col for col in status_share_cols})

completed_rate = (
    pos.assign(COMPLETED=(pos["NAME_CONTRACT_STATUS"] == "Completed").astype(int))
    .groupby("SK_ID_CURR")["COMPLETED"]
    .mean()
    .rename("POS_COMPLETED_RATE")
    .reset_index()
)

pos_features = pos_overall.copy()
for block in [pos_count, pos_late_rate, status_share, completed_rate]:
    pos_features = pos_features.merge(block, on="SK_ID_CURR", how="left")

print("POS features:", pos_features.shape)

POS features: (337252, 54)


In [11]:
# Merge POS features and create the has-history flag
application_model = application_model.merge(pos_features, on="SK_ID_CURR", how="left")
application_model["HAS_POS_HISTORY"] = application_model["POS_COUNT"].notna().astype(int)

print("Model06 shape:", application_model.shape)

Model06 shape: (307511, 274)


In [12]:
# Confirm shape, ID counts, and missingness before feature engineering
print("Installment rows:", len(installments))
print("Unique applicants:", installments["SK_ID_CURR"].nunique())
print("Unique previous loans:", installments["SK_ID_PREV"].nunique())
print("Missing values:")
display(installments.isna().sum())

Installment rows: 13605401
Unique applicants: 339587
Unique previous loans: 997752
Missing values:


,0
SK_ID_PREV,0
SK_ID_CURR,0
NUM_INSTALMENT_VERSION,0
NUM_INSTALMENT_NUMBER,0
DAYS_INSTALMENT,0
DAYS_ENTRY_PAYMENT,2905
AMT_INSTALMENT,0
AMT_PAYMENT,2905


In [13]:
# Payment ratio: amount actually paid / amount scheduled
installments["INSTALLMENT_PAYMENT_RATIO"] = (
    installments["AMT_PAYMENT"] / installments["AMT_INSTALMENT"].replace(0, np.nan)
)

# Positive INSTALLMENT_PAYMENT_DIFF means the borrower paid less than the scheduled installment amount.
installments["INSTALLMENT_PAYMENT_DIFF"] = installments["AMT_INSTALMENT"] - installments["AMT_PAYMENT"]

# Days past due: positive means payment happened after the scheduled date
installments["DAYS_PAYMENT_DELAY"] = installments["DAYS_ENTRY_PAYMENT"] - installments["DAYS_INSTALMENT"]
installments["DAYS_PAYMENT_DELAY_POSITIVE"] = installments["DAYS_PAYMENT_DELAY"].clip(lower=0)

# Days early: positive means payment happened before the scheduled date
installments["DAYS_PAID_EARLY"] = (-installments["DAYS_PAYMENT_DELAY"]).clip(lower=0)

# Late payment flag
installments["LATE_PAYMENT"] = (installments["DAYS_PAYMENT_DELAY"] > 0).astype("int8")

# Underpayment flag
installments["UNDERPAYMENT"] = (installments["INSTALLMENT_PAYMENT_DIFF"] > 0).astype("int8")

In [14]:
# Verify no infinite ratios, check zero-denominator prevalence, sanity-check flag counts
print("Infinite payment ratios:", np.isinf(installments["INSTALLMENT_PAYMENT_RATIO"]).sum())
print("Zero AMT_INSTALMENT count:", (installments["AMT_INSTALMENT"] == 0).sum())
print("Late payments:", installments["LATE_PAYMENT"].sum())
print("Underpayments:", installments["UNDERPAYMENT"].sum())

display(installments[[
    "AMT_INSTALMENT", "AMT_PAYMENT", "INSTALLMENT_PAYMENT_RATIO", "INSTALLMENT_PAYMENT_DIFF",
    "DAYS_PAYMENT_DELAY_POSITIVE", "DAYS_PAID_EARLY", "LATE_PAYMENT", "UNDERPAYMENT"
]].head())

Infinite payment ratios: 0
Zero AMT_INSTALMENT count: 290
Late payments: 1146669
Underpayments: 1295493


,AMT_INSTALMENT,AMT_PAYMENT,INSTALLMENT_PAYMENT_RATIO,INSTALLMENT_PAYMENT_DIFF,DAYS_PAYMENT_DELAY_POSITIVE,DAYS_PAID_EARLY,LATE_PAYMENT,UNDERPAYMENT
0,6948.359863,6948.359863,1.000000,0.000000,0.0,7.0,0,0
1,1716.525024,1716.525024,1.000000,0.000000,0.0,-0.0,0,0
2,25425.000000,25425.000000,1.000000,0.000000,0.0,-0.0,0,0
3,24350.130859,24350.130859,1.000000,0.000000,0.0,8.0,0,0
4,2165.040039,2160.584961,0.997942,4.455078,17.0,0.0,1,1


In [15]:
# Applicant-level summary of payment behavior across all installments
installment_agg = (
    installments.groupby("SK_ID_CURR")
    .agg({
        "NUM_INSTALMENT_VERSION": ["nunique"],
        "NUM_INSTALMENT_NUMBER": ["max", "mean"],
        "INSTALLMENT_PAYMENT_RATIO": ["mean", "max", "min"],
        "INSTALLMENT_PAYMENT_DIFF": ["mean", "max", "sum"],
        "DAYS_PAYMENT_DELAY_POSITIVE": ["mean", "max", "sum"],
        "DAYS_PAID_EARLY": ["mean", "max", "sum"],
        "AMT_INSTALMENT": ["mean", "max", "sum"],
        "AMT_PAYMENT": ["mean", "max", "sum"],
        "LATE_PAYMENT": ["sum"],
        "UNDERPAYMENT": ["sum"]
    })
)
installment_agg.columns = ["INST_" + col[0] + "_" + col[1].upper() for col in installment_agg.columns]
installment_agg = installment_agg.reset_index()

display(installment_agg.head())

,SK_ID_CURR,INST_NUM_INSTALMENT_VERSION_NUNIQUE,INST_NUM_INSTALMENT_NUMBER_MAX,INST_NUM_INSTALMENT_NUMBER_MEAN,INST_INSTALLMENT_PAYMENT_RATIO_MEAN,INST_INSTALLMENT_PAYMENT_RATIO_MAX,INST_INSTALLMENT_PAYMENT_RATIO_MIN,INST_INSTALLMENT_PAYMENT_DIFF_MEAN,INST_INSTALLMENT_PAYMENT_DIFF_MAX,INST_INSTALLMENT_PAYMENT_DIFF_SUM,...,INST_DAYS_PAID_EARLY_MAX,INST_DAYS_PAID_EARLY_SUM,INST_AMT_INSTALMENT_MEAN,INST_AMT_INSTALMENT_MAX,INST_AMT_INSTALMENT_SUM,INST_AMT_PAYMENT_MEAN,INST_AMT_PAYMENT_MAX,INST_AMT_PAYMENT_SUM,INST_LATE_PAYMENT_SUM,INST_UNDERPAYMENT_SUM
0,100001,2,4,2.714286,1.0,1.0,1.0,0.0,0.0,0.0,...,36.0,62.0,5885.132324,17397.900391,4.119593e+04,5885.132324,17397.900391,4.119593e+04,1,0
1,100002,2,19,10.000000,1.0,1.0,1.0,0.0,0.0,0.0,...,31.0,388.0,11559.247070,53093.746094,2.196257e+05,11559.247070,53093.746094,2.196257e+05,0,0
2,100003,2,12,5.080000,1.0,1.0,1.0,0.0,0.0,0.0,...,14.0,179.0,64754.585938,560835.375000,1.618865e+06,64754.585938,560835.375000,1.618865e+06,0,0
3,100004,2,3,2.000000,1.0,1.0,1.0,0.0,0.0,0.0,...,11.0,23.0,7096.154785,10573.964844,2.128846e+04,7096.154785,10573.964844,2.128846e+04,0,0
4,100005,2,9,5.000000,1.0,1.0,1.0,0.0,0.0,0.0,...,37.0,213.0,6240.205078,17656.244141,5.616184e+04,6240.205078,17656.244141,5.616184e+04,1,0


In [16]:
# Total number of installment records per applicant
installment_count = (
    installments.groupby("SK_ID_CURR")
    .size()
    .rename("INST_INSTALLMENT_COUNT")
    .reset_index()
)

installment_agg = installment_agg.merge(installment_count, on="SK_ID_CURR", how="left")

In [17]:
# Proportion of an applicant's installments paid late
installment_late_rate = (
    installments.groupby("SK_ID_CURR")["LATE_PAYMENT"]
    .mean()
    .rename("INST_LATE_PAYMENT_RATE")
    .reset_index()
)

installment_agg = installment_agg.merge(installment_late_rate, on="SK_ID_CURR", how="left")

In [18]:
# Proportion of an applicant's installments underpaid
installment_underpayment_rate = (
    installments.groupby("SK_ID_CURR")["UNDERPAYMENT"]
    .mean()
    .rename("INST_UNDERPAYMENT_RATE")
    .reset_index()
)

installment_agg = installment_agg.merge(installment_underpayment_rate, on="SK_ID_CURR", how="left")

In [19]:
# Confirm the key ratio and rate features look sensible before merging
print("Payment ratio statistics:")
display(installment_agg[[
    "INST_INSTALLMENT_PAYMENT_RATIO_MEAN",
    "INST_INSTALLMENT_PAYMENT_RATIO_MAX",
    "INST_INSTALLMENT_PAYMENT_RATIO_MIN",
    "INST_LATE_PAYMENT_RATE",
    "INST_UNDERPAYMENT_RATE"
]].describe().T)

Payment ratio statistics:


,count,mean,std,min,25%,50%,75%,max
INST_INSTALLMENT_PAYMENT_RATIO_MEAN,339575.0,1.360330,28.426548,0.333333,0.955224,1.000000,1.000000,8482.446289
INST_INSTALLMENT_PAYMENT_RATIO_MAX,339575.0,13.295465,839.615967,0.512192,1.000000,1.000000,1.000000,194250.000000
INST_INSTALLMENT_PAYMENT_RATIO_MIN,339575.0,0.597179,0.472363,0.000000,0.015000,1.000000,1.000000,1.969512
INST_LATE_PAYMENT_RATE,339587.0,0.074385,0.114490,0.000000,0.000000,0.017857,0.109375,1.000000
INST_UNDERPAYMENT_RATE,339587.0,0.082631,0.148284,0.000000,0.000000,0.000000,0.105263,1.000000


In [20]:
# Confirm the aggregation produced exactly one row per applicant
print("Installment feature table shape:", installment_agg.shape)
print("Unique applicants:", installment_agg["SK_ID_CURR"].nunique())
print("Duplicate applicant IDs:", installment_agg["SK_ID_CURR"].duplicated().sum())
print("One row per applicant:", len(installment_agg) == installment_agg["SK_ID_CURR"].nunique())

Installment feature table shape: (339587, 27)
Unique applicants: 339587
Duplicate applicant IDs: 0
One row per applicant: True


In [21]:
# List every engineered installment feature
installment_feature_cols = [col for col in installment_agg.columns if col != "SK_ID_CURR"]

print("Number of installment features:", len(installment_feature_cols))
for col in installment_feature_cols:
    print("-", col)

Number of installment features: 26
- INST_NUM_INSTALMENT_VERSION_NUNIQUE
- INST_NUM_INSTALMENT_NUMBER_MAX
- INST_NUM_INSTALMENT_NUMBER_MEAN
- INST_INSTALLMENT_PAYMENT_RATIO_MEAN
- INST_INSTALLMENT_PAYMENT_RATIO_MAX
- INST_INSTALLMENT_PAYMENT_RATIO_MIN
- INST_INSTALLMENT_PAYMENT_DIFF_MEAN
- INST_INSTALLMENT_PAYMENT_DIFF_MAX
- INST_INSTALLMENT_PAYMENT_DIFF_SUM
- INST_DAYS_PAYMENT_DELAY_POSITIVE_MEAN
- INST_DAYS_PAYMENT_DELAY_POSITIVE_MAX
- INST_DAYS_PAYMENT_DELAY_POSITIVE_SUM
- INST_DAYS_PAID_EARLY_MEAN
- INST_DAYS_PAID_EARLY_MAX
- INST_DAYS_PAID_EARLY_SUM
- INST_AMT_INSTALMENT_MEAN
- INST_AMT_INSTALMENT_MAX
- INST_AMT_INSTALMENT_SUM
- INST_AMT_PAYMENT_MEAN
- INST_AMT_PAYMENT_MAX
- INST_AMT_PAYMENT_SUM
- INST_LATE_PAYMENT_SUM
- INST_UNDERPAYMENT_SUM
- INST_INSTALLMENT_COUNT
- INST_LATE_PAYMENT_RATE
- INST_UNDERPAYMENT_RATE


In [22]:
# Add installment features onto the running Model06 dataset
application_model = application_model.merge(installment_agg, on="SK_ID_CURR", how="left")

print("Shape after installment features:", application_model.shape)
print("Unique applicants:", application_model["SK_ID_CURR"].nunique())
print("Duplicate applicant IDs:", application_model["SK_ID_CURR"].duplicated().sum())

Shape after installment features: (307511, 300)
Unique applicants: 307511
Duplicate applicant IDs: 0


In [23]:
# Create the has-history flag post-merge, as the single source of truth
application_model["HAS_INSTALLMENT_HISTORY"] = application_model["INST_INSTALLMENT_COUNT"].notna().astype(int)

print(application_model["HAS_INSTALLMENT_HISTORY"].value_counts())

HAS_INSTALLMENT_HISTORY
1    291643
0     15868
Name: count, dtype: int64


In [24]:
# Split into X and y for modeling
X = application_model.drop(columns=["TARGET", "SK_ID_CURR"])
y = application_model["TARGET"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (307511, 299)
y shape: (307511,)


In [25]:
# Same stratified split parameters used throughout the project
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print("Training shape:", X_train.shape)
print("Validation shape:", X_valid.shape)
print("\nTrain target distribution:")
print(y_train.value_counts(normalize=True))
print("\nValidation target distribution:")
print(y_valid.value_counts(normalize=True))

Training shape: (246008, 299)
Validation shape: (61503, 299)

Train target distribution:
TARGET
0    0.919271
1    0.080729
Name: proportion, dtype: float64

Validation target distribution:
TARGET
0    0.919272
1    0.080728
Name: proportion, dtype: float64


In [26]:
# Separate numeric vs categorical columns for preprocessing
numeric_features = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_features = X_train.select_dtypes(include=["object"]).columns.tolist()

print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))

Numeric features: 283
Categorical features: 16


In [27]:
# Median-impute numerics, most-frequent-impute + one-hot encode categoricals
numeric_transformer = Pipeline(steps=[("imputer", SimpleImputer(strategy="median"))])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

In [28]:
# Same hyperparameters as every prior experiment for a fair feature-only comparison
xgb_model = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="auc",
    tree_method="hist",
    random_state=42,
    n_jobs=-1
)

In [29]:
# Combine preprocessing and model
xgb_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", xgb_model)
])

In [30]:
# Fit the pipeline on training data
print("Training XGBoost with Model06 + Installment features...")
xgb_pipeline.fit(X_train, y_train)
print("Training complete.")

Training XGBoost with Model06 + Installment features...
Training complete.


In [31]:
# Generate validation-set probability predictions
valid_proba = xgb_pipeline.predict_proba(X_valid)[:, 1]
print("Predictions generated.")

Predictions generated.


In [32]:
# Compute ROC-AUC and PR-AUC on the validation set
roc_auc = roc_auc_score(y_valid, valid_proba)
pr_auc = average_precision_score(y_valid, valid_proba)

print("XGBOOST + MODEL06 + INSTALLMENTS")
print(f"ROC-AUC: {roc_auc:.4f}")
print(f"PR-AUC:  {pr_auc:.4f}")

XGBOOST + MODEL06 + INSTALLMENTS
ROC-AUC: 0.7859
PR-AUC:  0.2864


In [33]:
# Measure improvement over the locked Model06 result
MODEL06_ROC_AUC = 0.7833
MODEL06_PR_AUC = 0.2786

roc_change = roc_auc - MODEL06_ROC_AUC
pr_change = pr_auc - MODEL06_PR_AUC

print("IMPROVEMENT OVER MODEL06")
print(f"ROC-AUC change: {roc_change:+.4f}")
print(f"PR-AUC change:  {pr_change:+.4f}")

IMPROVEMENT OVER MODEL06
ROC-AUC change: +0.0026
PR-AUC change:  +0.0078


In [34]:
# Build a summary table of this experiment's results
installment_result = pd.DataFrame({
    "Experiment": ["XGBoost + Model06 + Installment Features"],
    "ROC-AUC": [roc_auc],
    "PR-AUC": [pr_auc],
    "ROC-AUC Change": [roc_change],
    "PR-AUC Change": [pr_change]
})

display(installment_result.style.format({
    "ROC-AUC": "{:.4f}",
    "PR-AUC": "{:.4f}",
    "ROC-AUC Change": "{:+.4f}",
    "PR-AUC Change": "{:+.4f}"
}))

,Experiment,ROC-AUC,PR-AUC,ROC-AUC Change,PR-AUC Change
0,XGBoost + Model06 + Installment Features,0.7859,0.2864,+0.0026,+0.0078


In [35]:
# Save this experiment's result to CSV
installment_result.to_csv(RESULTS_PATH + "installments_experiment.csv", index=False)
print("Experiment saved to:", RESULTS_PATH + "installments_experiment.csv")

Experiment saved to: /content/drive/MyDrive/RupeeRisk/installments_experiment.csv


In [36]:
# Install MLflow in this Colab session
!pip install mlflow -q
import mlflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 91.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 69.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 58.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.9/265.9 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 84.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.2/216.2 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.9/123.9 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132

In [37]:
# Point at the same tracking database used throughout the project
mlflow.set_tracking_uri("sqlite:////content/drive/MyDrive/RupeeRisk/mlflow.db")
mlflow.set_experiment("RupeeRisk")

<Experiment: artifact_location='/content/mlruns/1', creation_time=1787393296537, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1787393296537, lifecycle_stage='active', name='RupeeRisk', tags={}, trace_location=None, workspace='default'>

In [38]:
# Record this run's parameters and metrics
with mlflow.start_run(run_name="XGBoost_Installments"):
    mlflow.log_param("stage", "Model07 - Installments Feature Engineering")
    mlflow.log_param("model", "XGBoost")
    mlflow.log_param("builds_on", "Model06 application + previous + bureau + bureau balance + POS")
    mlflow.log_param("n_installment_features", len(installment_feature_cols))
    mlflow.log_param("derived_features", "payment_ratio, payment_diff, delay, early_payment, late_flag, underpayment")
    mlflow.log_param("scale_pos_weight", False)

    mlflow.log_param("n_estimators", 300)
    mlflow.log_param("learning_rate", 0.05)
    mlflow.log_param("max_depth", 6)
    mlflow.log_param("subsample", 0.8)
    mlflow.log_param("colsample_bytree", 0.8)

    mlflow.log_metric("roc_auc", roc_auc)
    mlflow.log_metric("pr_auc", pr_auc)
    mlflow.log_metric("roc_auc_change_vs_Model06", roc_change)
    mlflow.log_metric("pr_auc_change_vs_Model06", pr_change)

print("Model07 logged to MLflow.")

Model07 logged to MLflow.


In [39]:
# Pull every run logged so far for comparison
runs = mlflow.search_runs(experiment_names=["RupeeRisk"])
display(runs[["tags.mlflow.runName", "metrics.roc_auc", "metrics.pr_auc"]])

,tags.mlflow.runName,metrics.roc_auc,metrics.pr_auc
0,XGBoost_Installments,0.785949,0.286362
1,XGBoost_POS_CASH,0.783313,0.278581
2,XGBoost_Bureau,0.777585,0.274161
3,XGBoost_Previous_Application,0.775428,0.265853
4,XGBoost_Application_Features,0.769403,0.262725
5,XGBoost_scale_pos_weight,0.760000,0.249300
6,XGBoost_Baseline,0.761200,0.251600
7,Logistic_Regression_Baseline,0.750100,0.232600


In [40]:
# List every installment feature created in this notebook
print("Installment features created:")
for col in installment_feature_cols:
    print("-", col)
print("\nTotal installment features:", len(installment_feature_cols))

Installment features created:
- INST_NUM_INSTALMENT_VERSION_NUNIQUE
- INST_NUM_INSTALMENT_NUMBER_MAX
- INST_NUM_INSTALMENT_NUMBER_MEAN
- INST_INSTALLMENT_PAYMENT_RATIO_MEAN
- INST_INSTALLMENT_PAYMENT_RATIO_MAX
- INST_INSTALLMENT_PAYMENT_RATIO_MIN
- INST_INSTALLMENT_PAYMENT_DIFF_MEAN
- INST_INSTALLMENT_PAYMENT_DIFF_MAX
- INST_INSTALLMENT_PAYMENT_DIFF_SUM
- INST_DAYS_PAYMENT_DELAY_POSITIVE_MEAN
- INST_DAYS_PAYMENT_DELAY_POSITIVE_MAX
- INST_DAYS_PAYMENT_DELAY_POSITIVE_SUM
- INST_DAYS_PAID_EARLY_MEAN
- INST_DAYS_PAID_EARLY_MAX
- INST_DAYS_PAID_EARLY_SUM
- INST_AMT_INSTALMENT_MEAN
- INST_AMT_INSTALMENT_MAX
- INST_AMT_INSTALMENT_SUM
- INST_AMT_PAYMENT_MEAN
- INST_AMT_PAYMENT_MAX
- INST_AMT_PAYMENT_SUM
- INST_LATE_PAYMENT_SUM
- INST_UNDERPAYMENT_SUM
- INST_INSTALLMENT_COUNT
- INST_LATE_PAYMENT_RATE
- INST_UNDERPAYMENT_RATE

Total installment features: 26


In [41]:
# Print the overall before/after comparison for this stage
print("""
Model07 INSTALLMENTS FEATURE ENGINEERING COMPLETE

Model06 benchmark: ROC-AUC = 0.7833, PR-AUC = 0.2786
Model07 result:    ROC-AUC = {:.4f}, PR-AUC = {:.4f}
Change:            ROC-AUC = {:+.4f}, PR-AUC = {:+.4f}

Remaining: credit_card_balance, then feature refinement,
polynomial experiment, Optuna, SHAP.
""".format(roc_auc, pr_auc, roc_change, pr_change))


Model07 INSTALLMENTS FEATURE ENGINEERING COMPLETE

Model06 benchmark: ROC-AUC = 0.7833, PR-AUC = 0.2786
Model07 result:    ROC-AUC = 0.7859, PR-AUC = 0.2864
Change:            ROC-AUC = +0.0026, PR-AUC = +0.0078

Remaining: credit_card_balance, then feature refinement,
polynomial experiment, Optuna, SHAP.



In [42]:
# Free up memory from the large installments dataframe
del installments
gc.collect()
print("Installments dataframe removed from memory.")

Installments dataframe removed from memory.
